# 01 — Tool Calling Basics

*Level 5 — Agentic RAG*

## Objective
Give the model actual tools — vector search, document fetch, SQL, web search, graph lookup — over **real, open data**: TriviaQA (trivia questions + real Wikipedia articles + real answer aliases) for documents, and the open-source Northwind database for SQL.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "tools"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from agentic_common.dataset import prepare
from agentic_common.retrieval import DenseRetriever
from vector_tool import VectorTool, GetDocumentTool
from sql_tool import SqlTool

data = prepare()
print(f"corpus={len(data.corpus)} chunks, questions={len(data.questions)}")

corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)
vector_tool = VectorTool(retriever, data.corpus)
get_document_tool = GetDocumentTool(data)
sql_tool = SqlTool()


corpus=1195 chunks, questions=50


## Each tool, called directly (no agent yet)


In [3]:
results = vector_tool("sophomore year college", top_k=3)
for r in results:
    print(f"{r['score']:.3f}  {r['article_title']}: {r['text'][:100]}...")


0.698  Student: Freshmen, Sophomores, Juniors and Seniors (respectively), unless their undergraduate program calls f...
0.603  Student: with the Junior Cert. to the more independent learning environment associated with the senior cycle....
0.602  Student: a Senior is a student in the last (usually fourth) year of college, university, or high school. A st...


In [4]:
doc = get_document_tool(results[0]["article_title"])
print(f"Full document length: {len(doc)} chars")
print(doc[:300], "...")


Full document length: 32687 chars
A student or pupil is a learner, or someone who attends an educational institution. In Britain until about 2012, underage schoolchildren were always referred to as "pupils", while those attending university are termed "students". In the USA, and more recently also in Britain, the term "student" is a ...


In [5]:
result = sql_tool("How many products cost more than $50?")
print("SQL:", result["sql"])
print("Rows:", result["rows"])


SQL: SELECT COUNT(*) FROM Products WHERE UnitPrice > 50 LIMIT 50
Rows: [{'COUNT(*)': 7}]


## What I observed

Each tool is a plain callable with a narrow, single-purpose interface — `vector_search` returns *candidates*, `get_document` returns the *whole* source once you know which one you want, `sql_query` returns *structured rows*. An agent choosing between them (next notebook) is choosing between fundamentally different kinds of answers, not just different data sources.

## Next

[02 — Agent Planning](./02_agent_planning.ipynb)
